# From parallel archives to a document-isolated translation pilot

How do you keep two translations of the same EU act out of different splits, or preserve a “≤” when cleaning a message? This notebook identifies the actual training inputs and runs the Europarl reader, cleaning rules and an exact-overlap check on newly written examples. Run all cells with a Python 3 CPU kernel. The default run uses the standard library and makes no network requests.

This capstone was developed during [Google DeepMind’s AI Research Foundations course](https://github.com/google-deepmind/ai-foundations). The recorded pilot used four institutional text archives. The short examples here were written for this notebook and contain no corpus excerpts or reviewer responses. The last section can also inspect the real Europarl archive on your own machine.

## 1. Identify the source releases before extracting text

The four archives total about 2.85 GB in decimal units. The Croatian text export needs an additional XML alignment index for document IDs. Sizes and hashes below identify the **compressed** files used by the project.

| Corpus and source | Direction | Raw rows | Gemma training pairs | Document identity |
| --- | --- | ---: | ---: | --- |
| [Europarl v10, Philipp Koehn](https://www.statmt.org/europarl/) | Polish → English | 632,600 | 28,000 | Debate filename in the TSV |
| [JRC-Acquis v3.0, European Communities / JRC](https://joint-research-centre.ec.europa.eu/language-technology-resources/jrc-acquis_en), via [OPUS](https://opus.nlpl.eu/) | Polish → English | 1,610,962 | 72,000 | EU legal act (CELEX) from the alignment XML |
| [DGT-TM v2021, European Commission](https://joint-research-centre.ec.europa.eu/language-technology-resources/dgt-translation-memory_en), via OPUS | Croatian → English | 1,416,461 | 100,000 | CELEX restored using the supplementary OPUS index |
| [United Nations Parallel Corpus v1.0](https://www.un.org/dgacm/en/content/uncorpus), via OPUS | Russian → English | 25,173,398 | 100,000 | Matched English and Russian UN document paths |

These counts describe aligned records before cleaning and selected pairs after sampling. A pair can span more than one sentence. Consult each publisher’s reuse terms when working with the source text. The UN corpus is described by [Ziemski et al. (2016)](https://aclanthology.org/L16-1561/).

In [1]:
from collections import Counter
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
from tempfile import NamedTemporaryFile
from urllib.request import urlopen
import gzip
import hashlib
import html
import re
import unicodedata


@dataclass(frozen=True)
class Archive:
    url: str
    filename: str
    size: int
    sha256: str


opus = "https://object.pouta.csc.fi/OPUS-"
archives = {
    "europarl": Archive(
        "https://www.statmt.org/europarl/v10/training/europarl-v10.pl-en.tsv.gz",
        "Europarl-v10.pl-en.tsv.gz", 72_277_756,
        "10968af7f3b203d2c3b7f2c305f2a1baad417858c7a51444a25632f3a58f8f67",
    ),
    "jrc": Archive(
        opus + "JRC-Acquis/v3.0/moses/en-pl.txt.zip",
        "JRC-Acquis-v3.0.en-pl.txt.zip", 105_363_021,
        "7307d157478a125aefa85717c5e5156ba18eff19cf8e7e7b7e3eb15b5fba5dcb",
    ),
    "dgt": Archive(
        opus + "DGT/v2021/moses/en-hr.txt.zip",
        "DGT-v2021.en-hr.txt.zip", 81_466_113,
        "7f6c14973220f2890819b4742c1f64711ed01c9526c8ae036338d4502b654210",
    ),
    "unpc": Archive(
        opus + "UNPC/v1.0/moses/en-ru.txt.zip",
        "UNPC-v1.0.en-ru.txt.zip", 2_589_707_636,
        "6b97f111ecfa67f2534cb699441c91b4eb77e6e9aee31f2bdac40e993213683b",
    ),
    "dgt_index": Archive(
        opus + "DGT/v2021/xml/en-hr.xml.gz",
        "DGT-v2021.en-hr.xml.gz", 9_448_896,
        "8f26ba44df964029ef859f77d1f4164599c67ddab98cf3bbeb9ff35ca12adff5",
    ),
}
print("Four text archives:", f"{sum(a.size for k, a in archives.items() if k != 'dgt_index'):,} bytes")
print("Additional DGT alignment index:", f"{archives['dgt_index'].size:,} bytes")

Four text archives: 2,848,814,526 bytes
Additional DGT alignment index: 9,448,896 bytes


## 2. Preserve document IDs while reading pairs

The Europarl archive has eight tab-separated fields. The source, target and debate filename are the first three. Speaker and affiliation fields stay out of the translation input. Seven pinned line numbers in this particular release have an extra tab on the Polish side. Five have an empty leading field and two split a list number from its text. The parser repairs only these known layouts and raises on unexpected rows.

The OPUS archives use a different representation. The pilot matched the source and English Moses files in alignment order, counting XML links with nonempty segment IDs on both sides. For DGT, both strings of all 1,416,461 rows were checked against the original OPUS sentence XML to validate the document mapping. JRC and DGT share the same canonical CELEX key for a legal act. Giving them separate corpus-specific keys would miss that overlap.

In [2]:
@dataclass(frozen=True)
class Pair:
    document: str
    source: str
    target: str


repair_prefixes = {
    32236: "", 44621: "", 52494: "", 109377: "", 365823: "",
    462440: "13.", 614854: "12.",
}
debate_filename = re.compile(r"ep-\d{2}-\d{2}-\d{2}-[a-zA-Z0-9-]+\.txt")


def read_europarl_row(line_number, line):
    fields = line.rstrip("\r\n").split("\t")
    if line_number in repair_prefixes:
        if len(fields) != 9 or fields[0].strip() != repair_prefixes[line_number]:
            raise ValueError(f"Unexpected repair layout at Europarl row {line_number}")
        fields[:2] = [" ".join(field.strip() for field in fields[:2] if field.strip())]
    if len(fields) != 8 or not debate_filename.fullmatch(fields[2]):
        raise ValueError(f"Unexpected Europarl schema at row {line_number}")
    return Pair("europarl:" + fields[2], fields[0], fields[1])


def synthetic_row(source, target, filename):
    fields = [source, target, filename, "1", "1", "", "", ""]
    return "\t".join(fields)


train_file = "ep-00-01-01-demo1.txt"
validation_file = "ep-00-01-10-demo2.txt"
test_file = "ep-00-01-07-demo1.txt"
lines = [
    (462440, synthetic_row("13.\tGórna granica to 8 €.", "13. The upper limit is €8.", train_file)),
    (2, synthetic_row("Zostaw znaki + oraz &le;.", "Keep the + and &le; signs.", train_file)),
    (3, synthetic_row("Odbierz żeton w recepcji.", "Collect a token at reception.", test_file)),
    (4, synthetic_row("Proszę podać krótszą trasę.", "Please give the shorter route.", validation_file)),
    (5, synthetic_row("Weź żeton w recepcji.", "Collect a token at reception.", train_file)),
    (6, synthetic_row("Bank", "bank", train_file)),
    (7, synthetic_row("Pusty wpis.", "", train_file)),
]
parsed = [read_europarl_row(number, line) for number, line in lines]
assert parsed[0].source.startswith("13. Górna")
print("Repaired example:", parsed[0])

Repaired example: Pair(document='europarl:ep-00-01-01-demo1.txt', source='13. Górna granica to 8 €.', target='13. The upper limit is €8.')


## 3. Keep the text used for training separate from the key used to find duplicates

The pilot unescaped HTML entities and collapsed whitespace in the training strings. It kept source spelling, case and punctuation. For comparison it used Unicode NFKC normalization, case folding and whitespace normalization. The original quality filter rejected empty sides, known corruption markers, pairs with a side lacking letters, normalized identical sides, sides over 120 words and lengths with a ratio above 3.5. Digit-only amounts therefore need context; an amount inside a sentence can pass.

In [3]:
def clean(text):
    return " ".join(html.unescape(text).split())


def comparison_key(text):
    return " ".join(unicodedata.normalize("NFKC", text).casefold().split())


def rejection_reason(pair):
    sides = (pair.source, pair.target)
    if not all(sides):
        return "empty side"
    if any(marker in side for side in sides for marker in ("�", "愦灭;")):
        return "corrupt text"
    if any(not any(char.isalpha() for char in side) for side in sides):
        return "no letters"
    lengths = [len(side.split()) for side in sides]
    if max(lengths) > 120 or max(lengths) > 3.5 * min(lengths):
        return "word length or ratio"
    if comparison_key(pair.source) == comparison_key(pair.target):
        return "identical sides"
    return None


cleaned = [Pair(row.document, clean(row.source), clean(row.target)) for row in parsed]
eligible = [row for row in cleaned if rejection_reason(row) is None]
assert {"€", "+", "≤"} <= set("".join(row.source for row in eligible))
print("Filter decisions:", dict(Counter(rejection_reason(row) or "eligible" for row in cleaned)))
print("Symbols after cleaning:", cleaned[1].source)

Filter decisions: {'eligible': 5, 'identical sides': 1, 'empty side': 1}
Symbols after cleaning: Zostaw znaki + oraz ≤.


## 4. Split documents, then protect held-out text

The pilot hashed each canonical document ID with seed 42 into 100 buckets: five each for test and validation, the remaining 90 for training. Bucket percentages govern document assignment. The final number of **pairs** in each split comes from per-corpus quotas. Earlier splits get priority in the order test, validation, train.

The example below isolates the exact-overlap step. Two different synthetic documents contain the same English line; a document-only check would miss it. The full build also deduplicated whole pairs within each split, compared English digit-normalized templates and screened long same-side passages for near duplicates (at least 12 words, five-word shingles, Jaccard similarity ≥0.80).

In [4]:
def document_split(document):
    seed_material = f"slavic-pilot-v1|42|doc|{document}".encode("utf-8")
    bucket = int.from_bytes(hashlib.sha256(seed_material).digest()[:8], "big") % 100
    if bucket < 5:
        return "test"
    if bucket < 10:
        return "validation"
    return "train"


def text_keys(rows):
    return {comparison_key(side) for row in rows for side in (row.source, row.target)}


split_order = ("test", "validation", "train")
candidates = {
    split: [row for row in eligible if document_split(row.document) == split]
    for split in split_order
}
assert text_keys(candidates["test"]) & text_keys(candidates["train"]) == {
    comparison_key("Collect a token at reception.")
}

kept = {}
prior_text = set()
for split in split_order:
    kept[split] = [
        row for row in candidates[split]
        if comparison_key(row.source) not in prior_text
        and comparison_key(row.target) not in prior_text
    ]
    prior_text.update(text_keys(kept[split]))
    print(f"{split:10} {len(candidates[split])} candidates → {len(kept[split])} kept")

for a, b in combinations(split_order, 2):
    assert {row.document for row in kept[a]}.isdisjoint(row.document for row in kept[b])
    assert text_keys(kept[a]).isdisjoint(text_keys(kept[b]))
assert sum(map(len, kept.values())) == 4
print("All three split comparisons passed the document and cross-side exact-text checks.")

test       1 candidates → 1 kept
validation 1 candidates → 1 kept
train      3 candidates → 2 kept
All three split comparisons passed the document and cross-side exact-text checks.


## 5. Counts from the completed experiment

The original pilot scanned the archives into deterministic candidate reservoirs, applied quality and overlap filters, filled per-corpus quotas and shuffled each final split. It contained 100,000 training, 1,000 validation and 1,000 test pairs **per training language**. Ukrainian contributed zero training pairs. The full Gemma training release kept 299,424 pilot rows and replaced 575 corrupt Croatian pairs and one Russian staffing table whose cells had run together. Each replacement was from a document already represented in training. The 6,000 official validation/test rows stayed frozen, including possible DGT artifacts.

The final release reported zero shared official document IDs across train and the two held-out splits. It also reported zero normalized training-text matches, in either direction, against the official holdouts, 1,012 Ukrainian FLORES devtest rows, 72 historical everyday messages, 16 stress prompts and 72 reviewed B01 pairs. Those are exact-match results; paraphrases may remain. B01 and the historical 72-message benchmark are distinct.

Gemma used each of the final 300,000 training pairs in full. Its native chat sequences had a median of 88 tokens, a 99th percentile of 291 and a maximum of 2,352. Token lengths depend on the model and serialization. The original pilot’s word filter and an earlier small-Gemma byte ceiling serve different purposes. The full Gemma run removed that byte ceiling.

## Optional: acquire and check a source archive

Leave the tuple empty for the offline demonstration. To audit the complete Polish parliamentary file, change it to `("europarl",)` on a machine with network access and enough disk space. Each selected archive is saved under `slm-corpus-cache/` relative to the working directory. The UN archive alone is 2.59 GB. Files must match both the recorded length and SHA-256 before use; an unexpected download cannot overwrite the cache.

Selecting Europarl checks the entire archive against its schema and reports raw and nonempty pair counts without printing any corpus text. Other selections verify the archive bytes only. For the original 306,000-row pilot, the repository includes the [full builder](../pipeline/build_pilot.py) with the corpus readers and reservoir sampling, and the [independent verifier](../pipeline/verify_pilot.py) with the full duplicate audit. Follow the [README](../README.md) to run that pipeline. The later Gemma training release additionally used the 576 replacements described above.

In [5]:
def verify(path, archive):
    if path.stat().st_size != archive.size:
        raise ValueError(f"Unexpected size for {archive.filename}")
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != archive.sha256:
        raise ValueError(f"Unexpected SHA-256 for {archive.filename}")


def acquire(archive, directory):
    directory.mkdir(parents=True, exist_ok=True)
    destination = directory / archive.filename
    if destination.exists():
        verify(destination, archive)
        return destination
    with NamedTemporaryFile(dir=directory, delete=False) as temporary:
        path = Path(temporary.name)
    try:
        with urlopen(archive.url, timeout=60) as response, path.open("wb") as out:
            for block in iter(lambda: response.read(1024 * 1024), b""):
                if out.tell() + len(block) > archive.size:
                    raise ValueError(f"Download exceeded expected size for {archive.filename}")
                out.write(block)
        verify(path, archive)
        path.replace(destination)
    finally:
        path.unlink(missing_ok=True)
    return destination


def inspect_europarl(path):
    total = nonempty = 0
    with gzip.open(path, "rt", encoding="utf-8", newline="") as source:
        for total, line in enumerate(source, 1):
            pair = read_europarl_row(total, line)
            nonempty += bool(pair.source.strip() and pair.target.strip())
    if (total, nonempty) != (632_600, 630_768):
        raise ValueError(f"Unexpected Europarl counts: {total:,} raw, {nonempty:,} nonempty")
    print(f"Europarl: {total:,} raw rows; {nonempty:,} pairs with two nonempty sides.")


download_archives = ()  # Example: ("europarl",)
for key in download_archives:
    local_file = acquire(archives[key], Path("slm-corpus-cache"))
    print(f"{local_file.name}: size and SHA-256 verified")
    if key == "europarl":
        inspect_europarl(local_file)